In [1]:
import pyomo.environ as pyo
import pandas as pd
import numpy as np
import math
from pyomo.contrib.parmest.experiment import Experiment
from uncertainties import quantify_propagate_uncertainty
import pyomo.dae as dae
from pyomo.contrib.parmest import parmest
import idaes

## Validate the results of the Williams-Otto process plant

In [2]:
def solve_williams_otto_plant(param_values):
    """
    Validates the optimal solutions of the Williams-Otto process plant reported 
    in Biegler LT. Nonlinear Programming: Concepts, Algorithms, and Applications 
    to Chemical Processes. Society for Industrial and Applied Mathematics (2010).

    Parameters
    ----------
    param_values: dict,
        Value of the Arrhenius kinetic parameters
    """
    model = pyo.ConcreteModel()

    # define the global parameters
    rho = 50

    # define the reaction numbers
    reaction_number = [1, 2, 3]
    
    # define the model parameters
    model.alpha_1 = pyo.Var(bounds=(0, None),)
    model.alpha_1.fix(param_values["alpha_1"])
    
    model.alpha_2 = pyo.Var(bounds=(0, None),)
    model.alpha_2.fix(param_values["alpha_2"])
    
    model.alpha_3 = pyo.Var(bounds=(0, None),)
    model.alpha_3.fix(param_values["alpha_3"])
    
    model.E1 = pyo.Var(bounds=(0, None),)
    model.E1.fix(param_values["E1"])
    
    model.E2 = pyo.Var(bounds=(0, None),)
    model.E2.fix(param_values["E2"])
    
    model.E3 = pyo.Var(bounds=(0, None),)
    model.E3.fix(param_values["E3"])
    
    # define the decision variables
    model.V = pyo.Var(bounds=(0.03, 0.1), initialize=0.06,)
    model.T = pyo.Var(bounds=[5.8, 6.8], initialize=5.8,)
    model.FA = pyo.Var(bounds=(1, None), initialize=10,)
    model.FB = pyo.Var(bounds=(1, None), initialize=20,)
    model.eta = pyo.Var(bounds=(0, 1), initialize=0.1,)
    
    # define the components
    components = ['A','B','C','E','P','G']
    
    # add the effluent stream of the reactor
    model.Feff = pyo.Var(components, within=pyo.NonNegativeReals, initialize={'A':10,'B':30,'C':3,'E':3,'P':5,'G':1},)
    model.Feff_sum = pyo.Var(within=pyo.PositiveReals, initialize=52,)
    
    # add the product, waste, purge, and recycle streams
    model.FP = pyo.Var(bounds=(0, 4.763), initialize=0.5,)
    model.FG = pyo.Var(within=pyo.NonNegativeReals, initialize=1,)
    model.Fpurge = pyo.Var(within=pyo.NonNegativeReals, initialize=0,)
    model.FR = pyo.Var(components, within=pyo.NonNegativeReals, initialize={'A':10,'B':30,'C':3,'E':3,'P':0.3,'G':0},)
    
    # add the mass fractions
    model.X = pyo.Var(components, bounds=(0, 1), initialize={'A':0.3,'B':0.696,'C':1e-3,'E':1e-3,'P':1e-3,'G':1e-3},)

    # define the reparameterized temperature
    model.T_reparam = pyo.Var(bounds=[1/6.8, 1/5.8],)
    
    # add the temperature constraints
    model.temp_rule = pyo.Constraint(
        expr = model.T_reparam == 1 / model.T
    )
    
    # add the rate constants
    model.k = pyo.Var(reaction_number, bounds=(0, None), initialize={1:6.18, 2:15.2, 3:10.2},)
    model.k_reparam = pyo.Var(reaction_number, bounds=(0, None),)
    
    # add the reaction rates
    model.r = pyo.Var(reaction_number, bounds=(0, None),)
    
    # calculate the reparameterized rate constants
    def k_reparam_rule(model, i):
        if i == 1:
            return model.k_reparam[i] == model.alpha_1 - model.E1 * model.T_reparam
        elif i == 2:
            return model.k_reparam[i] == model.alpha_2 - model.E2 * model.T_reparam
        else:
            return model.k_reparam[i] == model.alpha_3 - model.E3 * model.T_reparam
            
    model.k_reparam_eq = pyo.Constraint(
        reaction_number, rule=k_reparam_rule
    )
    
    # calculate the original rate constants
    def k_rule(model, i):
        return model.k[i] == pyo.exp(model.k_reparam[i])
    
    model.k_eq = pyo.Constraint(
        reaction_number, rule=k_rule
    )
    
    # define the rates of reactions
    def reaction_rate_rule(model, i):
        if i == 1:
            return model.r[i] == model.k[i] * model.X['A'] * model.X['B'] * model.V * rho
        elif i == 2:
            return model.r[i] == model.k[i] * model.X['C'] * model.X['B'] * model.V * rho
        else:
            return model.r[i] == model.k[i] * model.X['P'] * model.X['C'] * model.V * rho
    
    model.reaction_rate = pyo.Constraint(
        reaction_number, rule=reaction_rate_rule
    )
    
    # define the species mass balances
    model.A_bal = pyo.Constraint(
        expr = model.Feff['A'] == model.FA + model.FR['A'] - model.r[1]
    )
    
    model.B_bal = pyo.Constraint(
        expr = model.Feff['B'] == model.FB + model.FR['B'] - (model.r[1] + model.r[2])
    )
    
    model.C_bal = pyo.Constraint(
        expr = model.Feff['C'] == model.FR['C'] + 2 * model.r[1] - 2 * model.r[2] - model.r[3]
    )
    
    model.E_bal = pyo.Constraint(
        expr = model.Feff['E'] == model.FR['E'] + 2 * model.r[2]
    )
    
    model.P_bal = pyo.Constraint(
        expr = model.Feff['P'] == 0.1 * model.FR['E'] + model.r[2] - 0.5 * model.r[3]
    )
    
    model.G_bal = pyo.Constraint(
        expr = model.Feff['G'] == 1.5 * model.r[3]
    )
    
    # calculate the total effluent flow
    model.total_effluent = pyo.Constraint(
        expr = model.Feff_sum == sum(model.Feff[j] for j in components)
    )
    
    # add the constraints on the total effluent flow
    model.total_effluent_constraint = pyo.Constraint(
        expr = model.Feff_sum >= 1
    )
    
    # ensure mass fraction balance
    def mole_frac_rule(model, j):
        return model.Feff[j] == model.Feff_sum * model.X[j]
    
    model.mole_frac = pyo.Constraint(components, rule=mole_frac_rule)
    
    # calculate the product flowrate 
    model.product = pyo.Constraint(
        expr = model.FP == model.Feff['P'] - 0.1 * model.Feff['E']
    )
    
    # calculate the flowrate of the waste
    model.waste = pyo.Constraint(
        expr = model.FG == model.Feff['G']
    )
    
    # calculate the flowrate of the purge stream
    model.purge = pyo.Constraint(
        expr = model.Fpurge == model.eta * (model.Feff['A'] + model.Feff['B'] + model.Feff['C'] + 1.1 * model.Feff['E'])
    )
    
    # calculate the flowrate of the recycle stream
    def recycle_rule(model, j):
        if j == "G":
            return model.FR[j] == 0
        elif j == "P":
            return model.FR[j] == (1 - model.eta) * 0.1 * model.Feff["E"]
        else:
            return model.FR[j] == (1 - model.eta) * model.Feff[j]
    
    model.recycle = pyo.Constraint(components, rule=recycle_rule)
    
    # define the objective function
    model.obj = pyo.Objective(
        expr = (2207*model.FP + 50*model.Fpurge - 168*model.FA - 252*model.FB - 2.22*model.Feff_sum 
                      - 84*model.FG - 60*model.V*rho)/(6 * model.V * rho),
        sense=pyo.maximize
    )
    
    # define the solver
    solver = pyo.SolverFactory('ipopt')
    
    # solve the model
    results = solver.solve(model, tee=True)
    
    # print the results
    print("ROI =", pyo.value(model.obj))
    for v in model.component_objects(pyo.Var, active=True):
        print("\n",v)
        for index in v:
            print(index, pyo.value(v[index]))

In [3]:
# define the ground-truth parameter values
true_params = {
    'alpha_1': 22.5109, 
    'alpha_2': 28.5851, 
    'alpha_3': 36.8035, 
    'E1': 120, 
    'E2': 150, 
    'E3': 200
}

solve_williams_otto_plant(true_params)

Ipopt 3.13.2: 

******************************************************************************
This program contains Ipopt, a library for large-scale nonlinear optimization.
 Ipopt is released as open source code under the Eclipse Public License (EPL).
         For more information visit http://projects.coin-or.org/Ipopt

This version of Ipopt was compiled from source code available at
    https://github.com/IDAES/Ipopt as part of the Institute for the Design of
    Advanced Energy Systems Process Systems Engineering Framework (IDAES PSE
    Framework) Copyright (c) 2018-2019. See https://github.com/IDAES/idaes-pse.

This version of Ipopt was compiled using HSL, a collection of Fortran codes
    for large-scale scientific computation.  All technical papers, sales and
    publicity material resulting from use of the HSL codes within IPOPT must
    contain the following acknowledgement:
        HSL, a collection of Fortran codes for large-scale scientific
        computation. See http://

## Solve the plant design problem at the estimated parameter values

In [4]:
# define the estimated parameter values
estimated_params = {
    "alpha_1": 22.6123,
    "alpha_2": 26.9903,
    "alpha_3": 34.1808,
    "E1": 120.7264,
    "E2": 141.1293,
    "E3": 184.6114,
}

solve_williams_otto_plant(estimated_params)

Ipopt 3.13.2: 

******************************************************************************
This program contains Ipopt, a library for large-scale nonlinear optimization.
 Ipopt is released as open source code under the Eclipse Public License (EPL).
         For more information visit http://projects.coin-or.org/Ipopt

This version of Ipopt was compiled from source code available at
    https://github.com/IDAES/Ipopt as part of the Institute for the Design of
    Advanced Energy Systems Process Systems Engineering Framework (IDAES PSE
    Framework) Copyright (c) 2018-2019. See https://github.com/IDAES/idaes-pse.

This version of Ipopt was compiled using HSL, a collection of Fortran codes
    for large-scale scientific computation.  All technical papers, sales and
    publicity material resulting from use of the HSL codes within IPOPT must
    contain the following acknowledgement:
        HSL, a collection of Fortran codes for large-scale scientific
        computation. See http://

## Set-up the process plant model for uncertainty propagation

In [5]:
def uncertain_williams_otto_plant():
    """
    Defines the process model of the Williams-Otto process plant 
    for uncertainty propagation

    Returns
    -------
    model: pyomo.ConcreteModel(),
        Pyomo model of the Williams-Otto process plant
    """
    model = pyo.ConcreteModel()

    # define the global parameters
    rho = 50

    # define the reaction numbers
    reaction_number = [1, 2, 3]
    
    # define the model parameters
    model.alpha_1 = pyo.Var(within=pyo.PositiveReals, initialize=10)
    
    model.alpha_2 = pyo.Var(within=pyo.PositiveReals, initialize=10)
    
    model.alpha_3 = pyo.Var(within=pyo.PositiveReals, initialize=10)
    
    model.E1 = pyo.Var(within=pyo.PositiveReals, initialize=50)
    
    model.E2 = pyo.Var(within=pyo.PositiveReals, initialize=50)
    
    model.E3 = pyo.Var(within=pyo.PositiveReals, initialize=50)

    # define the decision variables
    model.V = pyo.Var(bounds=(0.03, 0.1), initialize=0.06)
    model.T = pyo.Var(bounds=(5.8, 6.8), initialize=5.8)
    model.FA = pyo.Var(bounds=(1, 100), initialize=10)
    model.FB = pyo.Var(bounds=(1, 100), initialize=20)
    model.eta = pyo.Var(bounds=(0, 1), initialize=0.1)
    
    # define the components
    components = ['A','B','C','E','P','G']
    
    # add the effluent stream of the reactor
    model.Feff = pyo.Var(components, within=pyo.NonNegativeReals, initialize={'A':10,'B':30,'C':3,'E':3,'P':5,'G':1})
    model.Feff_sum = pyo.Var(within=pyo.PositiveReals, initialize=52)
    
    # add the product, waste, purge, and recycle streams
    model.FP = pyo.Var(bounds=(0, 4.763), initialize=0.5)
    model.FG = pyo.Var(within=pyo.NonNegativeReals, initialize=1)
    model.Fpurge = pyo.Var(within=pyo.NonNegativeReals, initialize=1e-3)
    
    recycle_comps = ['A','B','C','E','P']
    model.FR = pyo.Var(recycle_comps, within=pyo.NonNegativeReals, initialize={'A':10,'B':30,'C':3,'E':3,'P':0.3,})
    
    # add the mass fractions
    model.X = pyo.Var(components, bounds=(0, 1), initialize={'A':0.3,'B':0.696,'C':1e-3,'E':1e-3,'P':1e-3,'G':1e-3})

    # define the reparameterized temperature
    model.T_reparam = pyo.Var(bounds=(1/6.8, 1/5.8),)
    
    # add the temperature constraints
    model.temp_rule = pyo.Constraint(
        expr = model.T_reparam == 1 / model.T
    )
    
    # add the rate constants
    model.k = pyo.Var(reaction_number, within=pyo.PositiveReals, initialize={1:6.18, 2:15.2, 3:10.2},)
    model.k_reparam = pyo.Var(reaction_number, within=pyo.PositiveReals,)
    
    # add the reaction rates
    model.r = pyo.Var(reaction_number, within=pyo.PositiveReals,)
    
    # calculate the reparameterized rate constants
    def k_reparam_rule(model, i):
        if i == 1:
            return model.k_reparam[i] == model.alpha_1 - model.E1 * model.T_reparam
        elif i == 2:
            return model.k_reparam[i] == model.alpha_2 - model.E2 * model.T_reparam
        else:
            return model.k_reparam[i] == model.alpha_3 - model.E3 * model.T_reparam
            
    model.k_reparam_eq = pyo.Constraint(
        reaction_number, rule=k_reparam_rule
    )
    
    # calculate the original rate constants
    def k_rule(model, i):
        return model.k[i] == pyo.exp(model.k_reparam[i])
    
    model.k_eq = pyo.Constraint(
        reaction_number, rule=k_rule
    )

    # define the rates of reactions
    def reaction_rate_rule(model, i):
        if i == 1:
            return model.r[i] == model.k[i] * model.X['A'] * model.X['B'] * model.V * rho
        elif i == 2:
            return model.r[i] == model.k[i] * model.X['C'] * model.X['B'] * model.V * rho
        else:
            return model.r[i] == model.k[i] * model.X['P'] * model.X['C'] * model.V * rho
    
    model.reaction_rate = pyo.Constraint(
        reaction_number, rule=reaction_rate_rule
    )
    
    # define the species mass balances
    model.A_bal = pyo.Constraint(
        expr = model.Feff['A'] == model.FA + model.FR['A'] - model.r[1]
    )
    
    model.B_bal = pyo.Constraint(
        expr = model.Feff['B'] == model.FB + model.FR['B'] - (model.r[1] + model.r[2])
    )
    
    model.C_bal = pyo.Constraint(
        expr = model.Feff['C'] == model.FR['C'] + 2 * model.r[1] - 2 * model.r[2] - model.r[3]
    )
    
    model.E_bal = pyo.Constraint(
        expr = model.Feff['E'] == model.FR['E'] + 2 * model.r[2]
    )
    
    model.P_bal = pyo.Constraint(
        expr = model.Feff['P'] == 0.1 * model.FR['E'] + model.r[2] - 0.5 * model.r[3]
    )
    
    model.G_bal = pyo.Constraint(
        expr = model.Feff['G'] == 1.5 * model.r[3]
    )
    
    # calculate the total effluent flow
    model.total_effluent = pyo.Constraint(
        expr = model.Feff_sum == sum(model.Feff[j] for j in components)
    )
    
    # add the constraints on the total effluent flow
    model.total_effluent_constraint = pyo.Constraint(
        expr = model.Feff_sum >= 1
    )
    
    # ensure mass fraction balance
    def mole_frac_rule(model, j):
        return model.Feff[j] == model.Feff_sum * model.X[j]
    
    model.mole_frac = pyo.Constraint(components, rule=mole_frac_rule)
    
    # calculate the product flowrate 
    model.product = pyo.Constraint(
        expr = model.FP == model.Feff['P'] - 0.1 * model.Feff['E']
    )

    # calculate the flowrate of the waste
    model.waste = pyo.Constraint(
        expr = model.FG == model.Feff['G']
    )
    
    # calculate the flowrate of the purge stream
    model.purge = pyo.Constraint(
        expr = model.Fpurge == model.eta * (model.Feff['A'] + model.Feff['B'] + model.Feff['C'] + 1.1 * model.Feff['E'])
    )
    
    # calculate the flowrate of the recycle stream
    def recycle_rule(model, j):
        if j == "P":
            return model.FR[j] == (1 - model.eta) * 0.1 * model.Feff["E"]
        else:
            return model.FR[j] == (1 - model.eta) * model.Feff[j]
    
    model.recycle = pyo.Constraint(recycle_comps, rule=recycle_rule)
    
    # define the objective function
    model.obj = pyo.Objective(
        expr = (2207*model.FP + 50*model.Fpurge - 168*model.FA - 252*model.FB - 2.22*model.Feff_sum 
                      - 84*model.FG - 60*model.V*rho)/(6 * model.V * rho),
        sense=pyo.maximize
    )

    return model

## Set-up the batch reactor model for parameter estimation

In [6]:
class BatchReactorExperiment(Experiment):
    """Creates and labels the Pyomo model of the batch reactor
    
    Parameters
    ----------
    data: Pandas.DataFrame or .csv file,
        Data containing the sample time and measured values of mass fractions 
    XA0: float,
        Initial mass fraction of species A
    temp_control: int, float, list, or dict,
        Constant or piecewise-linear profile of the reaction temperature (in R)
    const_temp: Boolean,
        Species if the batch reactor is a constant or variable temperature system
    time_points: pandas.Series or list,
        Experiment time corresponding to the temperature profile

    Returns
    -------
    m: annotated Pyomo model of the batch reactor
    """

    def __init__(self, XA0, temp_control, const_temp=False, data=None, time_points=None,):
        self.data = data
        self.XA0 = XA0
        self.temp_control = temp_control
        self.const_temp = const_temp
        self.time_points = time_points
        self.model = None

    def get_labeled_model(self):
        self.create_model()
        self.label_model()

        return self.model

    def create_model(self):
        if self.const_temp:
            self.model = const_temp_batch_reactor(self.XA0, self.temp_control)
        else:
            self.model = var_temp_batch_reactor(self.XA0, self.temp_control, self.time_points)

        return self.model

    def label_model(self):

        m = self.model
        
        meas_time_points = self.data["Time (hr)"]
    
        # label the measured variables
        m.experiment_outputs = pyo.Suffix(direction=pyo.Suffix.LOCAL)
        m.experiment_outputs.update(
            (m.XA[t], self.data["XA"][ind]) for ind, t in enumerate(meas_time_points)
        )
        m.experiment_outputs.update(
            (m.XB[t], self.data["XB"][ind]) for ind, t in enumerate(meas_time_points)
        )
        m.experiment_outputs.update(
            (m.XC[t], self.data["XC"][ind]) for ind, t in enumerate(meas_time_points)
        )
        m.experiment_outputs.update(
            (m.XP[t], self.data["XP"][ind]) for ind, t in enumerate(meas_time_points)
        )
        m.experiment_outputs.update(
            (m.XE[t], self.data["XE"][ind]) for ind, t in enumerate(meas_time_points)
        )
        m.experiment_outputs.update(
            (m.XG[t], self.data["XG"][ind]) for ind, t in enumerate(meas_time_points)
        )
        
        # add the measurement errors
        m.measurement_error = pyo.Suffix(direction=pyo.Suffix.LOCAL)
        m.measurement_error.update(
            (m.XA[t], 0.001) for t in meas_time_points
        )
        m.measurement_error.update(
            (m.XB[t], 0.001) for t in meas_time_points
        )
        m.measurement_error.update(
            (m.XC[t], 0.001) for t in meas_time_points
        )
        m.measurement_error.update(
            (m.XP[t], 0.01) for t in meas_time_points
        )
        m.measurement_error.update(
            (m.XE[t], 0.01) for t in meas_time_points
        )
        m.measurement_error.update(
            (m.XG[t], 0.01) for t in meas_time_points
        )

        # label the unknown parameters
        m.unknown_parameters = pyo.Suffix(direction=pyo.Suffix.LOCAL)
        m.unknown_parameters.update(
            (k, pyo.value(k)) for k in [m.alpha_1, m.alpha_2, m.alpha_3, m.E1, m.E2, m.E3]
        )

        return m

In [7]:
# load the design conditions for the variable-temperature profile
var_temp_design = pd.read_csv("condition_number_optimal_design.csv")
var_temp_profile = var_temp_design["Temperature (1/R)"]
var_temp_optimal_XA0 = var_temp_design["XA"][0]
var_time_vals = var_temp_design["Time (hr)"]

In [8]:
def var_temp_batch_reactor(XA0, temp_profile, time_points,):
    """
    Reformulates the variable temperature batch reactor model for parameter estimation

    Parameters
    ----------
    XA0: float,
        Initial mass fraction of species A
    temp_profile: pandas.Series or list,
        Temperature profile from optimal experimental design
    time_points: pandas.Series or list,
        Experiment time corresponding to the temperature profile

    Returns
    -------
    model: Pyomo model of the variable temperature batch reactor
    
    """
    model = pyo.ConcreteModel()

    # define sets
    reaction_number = [1, 2, 3]
    model.t = dae.ContinuousSet(bounds=[0, 3]) # hour

    # define the model parameters
    model.alpha_1 = pyo.Var(bounds=(0, None), initialize=10)
    model.alpha_2 = pyo.Var(bounds=(0, None), initialize=10)
    model.alpha_3 = pyo.Var(bounds=(0, None), initialize=10)
    model.E1 = pyo.Var(bounds=(0, None), initialize=50)
    model.E2 = pyo.Var(bounds=(0, None), initialize=50)
    model.E3 = pyo.Var(bounds=(0, None), initialize=50)
    
    # add the mass fraction variables
    model.XA = pyo.Var(model.t, bounds=(0, 1), initialize=XA0)
    model.XB = pyo.Var(model.t, bounds=(0, 1), initialize=1-XA0)
    model.XC = pyo.Var(model.t, bounds=(0, 1), initialize=0)
    model.XE = pyo.Var(model.t, bounds=(0, 1), initialize=0)
    model.XP = pyo.Var(model.t, bounds=(0, 1), initialize=0)
    model.XG = pyo.Var(model.t, bounds=(0, 1), initialize=0)

    # add the temperature variables
    model.T_reparam = pyo.Var(model.t, bounds=(0, 1))

    # add the rate constants
    model.k_reparam = pyo.Var(reaction_number, model.t, bounds=(0, None))
    model.k = pyo.Var(reaction_number, model.t, bounds=(0, None))
    
    # calculate the reparameterized rate constants
    def k_reparam_rule(m, i, t):
        if i == 1:
            return m.k_reparam[i, t] == m.alpha_1 - m.E1 * m.T_reparam[t]
        elif i == 2:
            return m.k_reparam[i, t] == m.alpha_2 - m.E2 * m.T_reparam[t]
        else:
            return m.k_reparam[i, t] == m.alpha_3 - m.E3 * m.T_reparam[t]
            
    model.k_reparam_eq = pyo.Constraint(
        reaction_number, model.t, rule=k_reparam_rule
    )
    
    # calculate the original rate constants
    def k_rule(m, i, t):
        return m.k[i, t] == pyo.exp(m.k_reparam[i, t])
    
    model.k_eq = pyo.Constraint(
        reaction_number, model.t, rule=k_rule
    )

    # add the differential equations for XA, XB, XC, XE, XP, and XG
    model.dXA = dae.DerivativeVar(model.XA, wrt=model.t)
    model.dXB = dae.DerivativeVar(model.XB, wrt=model.t)
    model.dXC = dae.DerivativeVar(model.XC, wrt=model.t)
    model.dXE = dae.DerivativeVar(model.XE, wrt=model.t)
    model.dXG = dae.DerivativeVar(model.XG, wrt=model.t)
    # model.dXP = dae.DerivativeVar(model.XP, wrt=model.t)

    @model.Constraint(model.t)
    def xa_rate_ode(m, t):
        if t == m.t.first():
            return pyo.Constraint.Skip
        return m.dXA[t] == - m.k[1, t] * m.XA[t] * m.XB[t]
    
    @model.Constraint(model.t)
    def xb_rate_ode(m, t):
        if t == m.t.first():
            return pyo.Constraint.Skip
        return m.dXB[t] == - (m.k[1, t] * m.XA[t] * m.XB[t] + m.k[2, t] * m.XB[t] * m.XC[t])
    
    @model.Constraint(model.t)
    def xc_rate_ode(m, t):
        if t == m.t.first():
            return pyo.Constraint.Skip
        return m.dXC[t] == 2 * m.k[1, t] * m.XA[t] * m.XB[t] - 2 * m.k[2, t] * m.XB[t] * m.XC[t] - m.k[3, t] * m.XC[t] * m.XP[t]
    
    @model.Constraint(model.t)
    def xe_rate_ode(m, t):
        if t == m.t.first():
            return pyo.Constraint.Skip
        return m.dXE[t] == 2 * m.k[2, t] * m.XB[t] * m.XC[t]
    
    @model.Constraint(model.t)
    def xg_rate_ode(m, t):
        if t == m.t.first():
            return pyo.Constraint.Skip
        return m.dXG[t] == 1.5 * m.k[3, t] * m.XC[t] * m.XP[t]

    # add the mass fraction constraint
    @model.Constraint(model.t)
    def sum_mass_fraction(m, t):
        return m.XA[t] + m.XB[t] + m.XC[t] + m.XE[t] + m.XG[t] + m.XP[t] == 1

    # fix the initial conditions
    t0 = model.t.first()
    model.XA_init = pyo.Constraint(expr=model.XA[t0] == XA0)
    model.XB_init = pyo.Constraint(expr=model.XB[t0] == 1 - model.XA[t0])
    model.XC_init = pyo.Constraint(expr=model.XC[t0] == 0.0)
    model.XE_init = pyo.Constraint(expr=model.XE[t0] == 0.0)
    model.XG_init = pyo.Constraint(expr=model.XG[t0] == 0.0)

    # discretize the model
    disc = pyo.TransformationFactory("dae.finite_difference")
    disc.apply_to(model, nfe=90, scheme="BACKWARD")

    # add the optimal temperature profile
    for indx, t in enumerate(time_points):
        model.T_reparam[t].fix(temp_profile[indx])
    
    return model

In [9]:
def const_temp_batch_reactor(XA0, temp):
    """
    Reformulates the constant temperature batch reactor model for parameter estimation

    Parameters
    ----------
    XA0: float,
        Initial mass fraction of species A
    temp: int or float,
        Constant reaction temperature in R

    Returns
    -------
    model: Pyomo model of the batch reactor
    
    """
    model = pyo.ConcreteModel()

    # define sets
    reaction_number = [1, 2, 3]
    model.t = dae.ContinuousSet(bounds=[0, 3]) # hour

    # define the model parameters
    model.alpha_1 = pyo.Var(bounds=(0, None), initialize=10)
    model.alpha_2 = pyo.Var(bounds=(0, None), initialize=10)
    model.alpha_3 = pyo.Var(bounds=(0, None), initialize=10)
    model.E1 = pyo.Var(bounds=(0, None), initialize=50)
    model.E2 = pyo.Var(bounds=(0, None), initialize=50)
    model.E3 = pyo.Var(bounds=(0, None), initialize=50)
    
    # add the mass fraction variables
    model.XA = pyo.Var(model.t, bounds=(0, 1), initialize=XA0)
    model.XB = pyo.Var(model.t, bounds=(0, 1), initialize=1-XA0)
    model.XC = pyo.Var(model.t, bounds=(0, 1), initialize=0)
    model.XE = pyo.Var(model.t, bounds=(0, 1), initialize=0)
    model.XP = pyo.Var(model.t, bounds=(0, 1), initialize=0)
    model.XG = pyo.Var(model.t, bounds=(0, 1), initialize=0)

    # add the temperature variable
    model.T_reparam = pyo.Var(bounds=(0, 1))
    model.T_reparam.fix(1/temp)

    # add the rate constants
    model.k_reparam = pyo.Var(reaction_number, bounds=(0, None))
    model.k = pyo.Var(reaction_number, bounds=(0, None))
    
    # calculate the reparameterized rate constants
    def k_reparam_rule(m, i):
        if i == 1:
            return m.k_reparam[i] == m.alpha_1 - m.E1 * m.T_reparam
        elif i == 2:
            return m.k_reparam[i] == m.alpha_2 - m.E2 * m.T_reparam
        else:
            return m.k_reparam[i] == m.alpha_3 - m.E3 * m.T_reparam
            
    model.k_reparam_eq = pyo.Constraint(
        reaction_number, rule=k_reparam_rule
    )
    
    # calculate the original rate constants
    def k_rule(m, i):
        return m.k[i] == pyo.exp(m.k_reparam[i])
    
    model.k_eq = pyo.Constraint(
        reaction_number, rule=k_rule
    )

    # add the differential equations for XA, XB, XC, XE, XP, and XG
    model.dXA = dae.DerivativeVar(model.XA, wrt=model.t)
    model.dXB = dae.DerivativeVar(model.XB, wrt=model.t)
    model.dXC = dae.DerivativeVar(model.XC, wrt=model.t)
    model.dXE = dae.DerivativeVar(model.XE, wrt=model.t)
    model.dXG = dae.DerivativeVar(model.XG, wrt=model.t)

    @model.Constraint(model.t)
    def xa_rate_ode(m, t):
        if t == m.t.first():
            return pyo.Constraint.Skip
        return m.dXA[t] == - m.k[1] * m.XA[t] * m.XB[t]
    
    @model.Constraint(model.t)
    def xb_rate_ode(m, t):
        if t == m.t.first():
            return pyo.Constraint.Skip
        return m.dXB[t] == - (m.k[1] * m.XA[t] * m.XB[t] + m.k[2] * m.XB[t] * m.XC[t])
    
    @model.Constraint(model.t)
    def xc_rate_ode(m, t):
        if t == m.t.first():
            return pyo.Constraint.Skip
        return m.dXC[t] == 2 * m.k[1] * m.XA[t] * m.XB[t] - 2 * m.k[2] * m.XB[t] * m.XC[t] - m.k[3] * m.XC[t] * m.XP[t]
    
    @model.Constraint(model.t)
    def xe_rate_ode(m, t):
        if t == m.t.first():
            return pyo.Constraint.Skip
        return m.dXE[t] == 2 * m.k[2] * m.XB[t] * m.XC[t]
    
    @model.Constraint(model.t)
    def xg_rate_ode(m, t):
        if t == m.t.first():
            return pyo.Constraint.Skip
        return m.dXG[t] == 1.5 * m.k[3] * m.XC[t] * m.XP[t]

    # add the mass fraction constraint
    @model.Constraint(model.t)
    def sum_mass_fraction(m, t):
        return m.XA[t] + m.XB[t] + m.XC[t] + m.XE[t] + m.XG[t] + m.XP[t] == 1

    # fix the initial conditions
    t0 = model.t.first()
    model.XA_init = pyo.Constraint(expr=model.XA[t0] == XA0)
    model.XB_init = pyo.Constraint(expr=model.XB[t0] == 1 - model.XA[t0])
    model.XC_init = pyo.Constraint(expr=model.XC[t0] == 0.0)
    model.XE_init = pyo.Constraint(expr=model.XE[t0] == 0.0)
    model.XG_init = pyo.Constraint(expr=model.XG[t0] == 0.0)

    # discretize the model
    disc = pyo.TransformationFactory("dae.finite_difference")
    disc.apply_to(model, nfe=60, scheme="BACKWARD")
    
    # define the solver
    solver = pyo.SolverFactory('ipopt')
    
    # solve the model
    results = solver.solve(model, tee=True)
    
    return model

In [10]:
# load the constant-temperature data
data_const_temp = pd.read_csv("batch_reactor_const_temp_low.csv")

# create an experiment class for the constant-temperature data
exp_const_temp = BatchReactorExperiment(
    data=data_const_temp, XA0=0.303,
    temp_control=5.8,
    const_temp=True,
)

In [11]:
# load the variable-temperature data
data_var_temp = pd.read_csv("batch_reactor_cond_optimal_design.csv")

# create an experiment class for the constant-temperature data
exp_var_temp = BatchReactorExperiment(
    data=data_var_temp, XA0=var_temp_optimal_XA0,
    temp_control=var_temp_profile,
    time_points=var_time_vals,
)

In [12]:
# create the experiment list
exp_list = [exp_const_temp, exp_var_temp]

## Set-up uncertainty propagation to the process plant

In [13]:
# define the uncertain parameters
uncertain_params = ["alpha_1", "alpha_2", "alpha_3", "E1", "E2", "E3"]

solver_options = {"tol": 1e-6}

# propagate the parameter uncertainty to the process design
propagation_results = quantify_propagate_uncertainty(exp_list, 
                                                     uncertain_williams_otto_plant, 
                                                     uncertain_params, "SSE_weighted", 
                                                     tee=True, 
                                                     solver_options=solver_options)

Ipopt 3.13.2: 

******************************************************************************
This program contains Ipopt, a library for large-scale nonlinear optimization.
 Ipopt is released as open source code under the Eclipse Public License (EPL).
         For more information visit http://projects.coin-or.org/Ipopt

This version of Ipopt was compiled from source code available at
    https://github.com/IDAES/Ipopt as part of the Institute for the Design of
    Advanced Energy Systems Process Systems Engineering Framework (IDAES PSE
    Framework) Copyright (c) 2018-2019. See https://github.com/IDAES/idaes-pse.

This version of Ipopt was compiled using HSL, a collection of Fortran codes
    for large-scale scientific computation.  All technical papers, sales and
    publicity material resulting from use of the HSL codes within IPOPT must
    contain the following acknowledgement:
        HSL, a collection of Fortran codes for large-scale scientific
        computation. See http://

In [14]:
# get the estimated parameters
propagation_results.theta

{'alpha_1': 22.604725821655055,
 'alpha_2': 26.875198668514525,
 'alpha_3': 33.79545843338095,
 'E1': 120.68776977402086,
 'E2': 140.47525183791845,
 'E3': 182.27039429890914}

In [15]:
# get the parameter uncertainty
propagation_results.cov

,alpha_1,alpha_2,alpha_3,E1,E2,E3
alpha_1,0.038717,0.077867,0.099047,0.227949,0.451758,0.582787
alpha_2,0.077867,3.533537,3.605446,0.447923,20.562296,20.994982
alpha_3,0.099047,3.605446,3.972644,0.578081,20.971285,23.199217
E1,0.227949,0.447923,0.578081,1.343271,2.597945,3.403808
E2,0.451758,20.562296,20.971285,2.597945,119.668849,122.127997
E3,0.582787,20.994982,23.199217,3.403808,122.127997,135.515899


In [16]:
# get the optimal plant design objective value
# get the estimated parameters
propagation_results.propagation_f

np.float64(8241.405392830988)